# 0. Imports

In [1]:
#!pip install -qq ipython numpy pandas scikit-learn statsmodels xgboost torch

In [2]:
import sys
import warnings
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import sklearn
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import ConvergenceWarning

import xgboost
from xgboost import XGBRegressor

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# --- New from Edward ---

from xgboost import XGBClassifier

from sklearn.preprocessing import LabelEncoder

import sys
feat_boost_relpath = "./FeatBoost-Fork/featboost" # Relative path from the notebook files.
if feat_boost_relpath not in sys.path:
    sys.path.append(feat_boost_relpath)
from feat_boost import FeatBoostClassifier

# --- New from Edward ---

In [3]:
!python --version
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)
print("feat_boost", "custom")

Python 3.10.18
numpy 2.0.1
pandas 2.3.2
scikit-learn 1.7.1
torch 2.6.0
feat_boost custom


In [4]:
# Extract all datasets from /data/ into a nested dictionary.
DFDICT = {
    subdir.name: {
        csv.stem: pd.read_csv(csv, dtype="string", low_memory=False)
        for csv in subdir.glob("*.csv")
    }
    for subdir in (Path.cwd() / "data").iterdir() if subdir.is_dir()
}

# 1. Preprocessing

### 1.1 Select and Join Datasets

In [5]:
# Here we select what data to use from what's available.

# Specify a set of columns from each dataset for selection
weatherCols = [
    "DATE",
    "HourlyDryBulbTemperature",
    "HourlyRelativeHumidity",
    "HourlyWindSpeed",
    "HourlyStationPressure",
    "HourlyVisibility"
]
electricityCols = [
    "UTC time",
    "Hour",
    "Demand",
]

# Select all 2024 weather and electrical data from those columns
DFW = DFDICT['weather']['MIA_2024'][weatherCols]
DFE = DFDICT['electricity']['FPL'][electricityCols]
DFE = DFE.iloc[ (i := 24 * (8*365 + 2 + 31*4 + 30*2)) : i + 24*366].copy()

In [6]:
# Eletrical grid observations are ALWAYS every hour on the hour.  
# Weather observations are temporally much messier.
# The output below shows the distribution of temporal differences 
#  across all 2024 weather observations.

_df = pd.DataFrame({"t": pd.to_datetime(DFW["DATE"])})
_df["delta t"] = (
    _df["t"].diff()
    .apply(lambda x: "" if pd.isna(x) else str(x).split()[-1])
)

print("Number of rows: ", len(_df), end='\n\n')
print(_df.head(10), end="\n\n")
with pd.option_context("display.max_rows", None):
    print(_df["delta t"].value_counts().sort_index().reset_index(name="Count"))

Number of rows:  13105

                    t   delta t
0 2024-01-01 00:53:00          
1 2024-01-01 01:00:00  00:07:00
2 2024-01-01 01:53:00  00:53:00
3 2024-01-01 02:53:00  01:00:00
4 2024-01-01 03:53:00  01:00:00
5 2024-01-01 04:00:00  00:07:00
6 2024-01-01 04:53:00  00:53:00
7 2024-01-01 05:53:00  01:00:00
8 2024-01-01 06:53:00  01:00:00
9 2024-01-01 07:00:00  00:07:00

     delta t  Count
0                 1
1   00:00:00     17
2   00:01:00     16
3   00:02:00    186
4   00:03:00     72
5   00:04:00     48
6   00:05:00     61
7   00:06:00    393
8   00:07:00   2820
9   00:08:00     40
10  00:09:00     42
11  00:10:00     52
12  00:11:00     54
13  00:12:00     41
14  00:13:00     45
15  00:14:00     42
16  00:15:00     38
17  00:16:00     36
18  00:17:00     50
19  00:18:00     35
20  00:19:00     35
21  00:20:00     36
22  00:21:00     45
23  00:22:00     38
24  00:23:00     33
25  00:24:00     25
26  00:25:00     28
27  00:26:00     35
28  00:27:00     30
29  00:28:00     27
30 

In [7]:
# Each dataset's time columns is replace by with another in local time
#  which is UTC-5 (the weather data needs no time adjustment).
_dfe, _dfw = (
    DFE.drop(columns=["UTC time"])
       .assign(t=pd.to_datetime(DFE["UTC time"]) - pd.Timedelta(hours=5)),
    DFW.drop(columns=["DATE"])
       .assign(t=pd.to_datetime(DFW["DATE"]))
)

# Merge both dataframes, joining records on their time measurement.
# CRITICALLY, the nearest weather obversation to each hour is naively taken.
# THIS IS BAD AND NEEDS TO BE CHANGED.
DF = pd.merge_asof(_dfe, _dfw, on="t", direction="nearest")
display(DF)

,Hour,Demand,t,HourlyDryBulbTemperature,HourlyRelativeHumidity,HourlyWindSpeed,HourlyStationPressure,HourlyVisibility
0,24,"11,815",2024-01-01 00:00:00,57,81,5,30.20,7.00
1,1,"11,254",2024-01-01 01:00:00,57,81,5,30.20,6.84
2,2,"11,109",2024-01-01 02:00:00,57,81,0,30.19,6.00
3,3,"10,861",2024-01-01 03:00:00,56,87,0,30.18,6.00
4,4,"10,736",2024-01-01 04:00:00,55,87,3,30.17,9.94
...,...,...,...,...,...,...,...,...
8779,19,"16,796",2024-12-31 19:00:00,75,84,0,29.94,9.94
8780,20,"15,515",2024-12-31 20:00:00,74,88,3,<NA>,10.00
8781,21,"14,348",2024-12-31 21:00:00,74,88,0,<NA>,10.00
8782,22,"13,391",2024-12-31 22:00:00,73,90,5,29.98,9.94


In [8]:
# # Count missing values, list unique weather data values.

# print(DF.shape)
# print()

# for c in [*electricityCols[1:], *weatherCols[1:]]:
#     print(f"{c:<25} {(DF[c].isna()).sum()}")
# print()

# for c in weatherCols[1:]:
#     print(f"{c}: Unique Values")
#     print(DF[c].unique().tolist())
#     print()

### 1.2 Convert Datatypes and Construct Features

In [9]:
# Convert to numeric dtypes and impute missing values.
# CRITICALLY, the nearest observation is imputed for each missing value again.
# THIS IS BAD AND NEEDS TO BE CHANGED.

DF["HourlyDryBulbTemperature"] = DF["HourlyDryBulbTemperature"].str[:2]
for c in weatherCols[1:]:
    converted = pd.to_numeric(DF[c], errors="coerce") 
    intFlag = (converted.dropna() % 1 == 0).all()
    imputed = converted.interpolate(method="nearest").ffill().bfill()
    DF[c] = imputed.astype("uint8" if intFlag else "float32")

converted = pd.to_numeric(DF["Demand"].str.replace(",", "", regex=False), errors="coerce")
DF["Demand"] = converted.interpolate(method="nearest").fillna(0).astype("uint16")
DF["Hour"] = pd.to_numeric(DF["Hour"]).astype("uint8")

display(DF)

,Hour,Demand,t,HourlyDryBulbTemperature,HourlyRelativeHumidity,HourlyWindSpeed,HourlyStationPressure,HourlyVisibility
0,24,11815,2024-01-01 00:00:00,57,81,5,30.200001,7.00
1,1,11254,2024-01-01 01:00:00,57,81,5,30.200001,6.84
2,2,11109,2024-01-01 02:00:00,57,81,0,30.190001,6.00
3,3,10861,2024-01-01 03:00:00,56,87,0,30.180000,6.00
4,4,10736,2024-01-01 04:00:00,55,87,3,30.170000,9.94
...,...,...,...,...,...,...,...,...
8779,19,16796,2024-12-31 19:00:00,75,84,0,29.940001,9.94
8780,20,15515,2024-12-31 20:00:00,74,88,3,29.940001,10.00
8781,21,14348,2024-12-31 21:00:00,74,88,0,29.980000,10.00
8782,22,13391,2024-12-31 22:00:00,73,90,5,29.980000,9.94


In [10]:
# Make sure the column name is *exactly* this:
col = "HourlyStationPressure"

s = pd.to_numeric(DF[col], errors="coerce")
print("Before: NaNs in HourlyStationPressure =", s.isna().sum())

s = s.interpolate(method="nearest").ffill().bfill()
print("After interpolate+ffill+bfill: NaNs =", s.isna().sum())

DF[col] = s.astype("float32")
print("In DF: NaNs in HourlyStationPressure =", DF[col].isna().sum())

Before: NaNs in HourlyStationPressure = 1
After interpolate+ffill+bfill: NaNs = 0
In DF: NaNs in HourlyStationPressure = 0


In [11]:
# Construct additional features from the selected columns

# AUTOREGRESSIVE FEATURES
# In this case each include each of the previous 3 hrs of measured demand
#  along with the previous day and week (24 and 168 hrs).
# ! The first h rows are blank for these features - here we backfill them. CHANGE
# ! Many more should be included, especially for the DL models.
lag_hours = [1, 2, 3, 24, 168]
for h in lag_hours:
    DF[f"lag_{h}"] = DF["Demand"].shift(h).bfill().astype("float32")
lagFeats = [f"lag_{h}" for h in lag_hours]

# CALENDAR FEATURES
# For now, just trig transformations on the hour of day and a one-hot encoding for weekends.
# ! Flags can be included for each day of the week, each month, holidays, etc.
DF["sin(h)"] = np.sin(2 * np.pi * DF["Hour"] / 24.0).astype("float32")
DF["cos(h)"] = np.cos(2 * np.pi * DF["Hour"] / 24.0).astype("float32")
DF["is_weekend"] = (DF["t"].dt.dayofweek >= 5).astype("int8")
DF = DF.drop(columns=["Hour"])
calendarFeats = ["sin(h)", "cos(h)", "is_weekend"]

# WEATHER FEATURES
# We don't extract any new features yet for this category.
weatherFeats = weatherCols[1:]  # skip DATE

# Define other convenient feature variables.
allFeats = lagFeats + calendarFeats + weatherFeats
target = "Demand"

# Drop rows missing any needed inputs (initial lags, any NAs)
DFmodel = DF.sort_values("t").reset_index(drop=True)
print("Model DF shape:", DFmodel.shape)

nan_count = DFmodel.isna().sum()
print(nan_count)

Model DF shape: (8784, 15)
Demand                      0
t                           0
HourlyDryBulbTemperature    0
HourlyRelativeHumidity      0
HourlyWindSpeed             0
HourlyStationPressure       0
HourlyVisibility            0
lag_1                       0
lag_2                       0
lag_3                       0
lag_24                      0
lag_168                     0
sin(h)                      0
cos(h)                      0
is_weekend                  0
dtype: int64


### 1.3 Data Splits

In [12]:
# Split out last 5 weeks of data for testing.

nTestDays = 35
testStartTime = DFmodel["t"].max() - pd.Timedelta(days=nTestDays)

DFtrain = DFmodel[DFmodel["t"] <  testStartTime].copy()
DFtest  = DFmodel[DFmodel["t"] >= testStartTime].copy()

X_train = DFtrain[allFeats].to_numpy(dtype=np.float32)
y_train = DFtrain[target].to_numpy(dtype=np.float32)
X_test  = DFtest[allFeats].to_numpy(dtype=np.float32)
y_test  = DFtest[target].to_numpy(dtype=np.float32)

print("Train size:  ", len(DFtrain))
print("Test size:   ", len(DFtest))
print("Train range: ", DFtrain["t"].min(), " - ", DFtrain["t"].max())
print("Test range:  ", DFtest["t"].min(),  " - ", DFtest["t"].max())

Train size:   7943
Test size:    841
Train range:  2024-01-01 00:00:00  -  2024-11-26 22:00:00
Test range:   2024-11-26 23:00:00  -  2024-12-31 23:00:00


### 1.4 Feature Selection using FeatBoost

In [15]:
# --- New from Edward ---

# FeatBoost
# Credits to Alsahaf et al. for developing FeatBoost.
# A custom fork is used here, with added bug fixes.

# Set up input data
print(allFeats)
X_fs = X_train
y_fs = y_train

# Set up estimator
fs_estimator = XGBRegressor(
    max_depth=10,
    learning_rate=0.1,
    n_estimators=20,
    objective="reg:squarederror",
    n_jobs=1,
    random_state=42,
)

'''
# TODO remove this later
fs_estimator = XGBClassifier(
    max_depth=3,
    learning_rate=0.1,
    n_estimators=20,
    silent=True,
    objective="binary:logistic",
    booster="gbtree",
    n_jobs=1,
    nthread=None,
    gamma=0,
    min_child_weight=1,
    max_delta_step=0,
    subsample=1,
    colsample_bytree=1,
    colsample_bylevel=1,
    reg_alpha=0,
    reg_lambda=1,
    scale_pos_weight=1,
    base_score=0.5,
    random_state=0,
    seed=42,
    missing=None
)
'''

# Set up feature selection method
# Note: "siso_ranking_size" (output size) must be <= #features in the input dataframe (input size).
#       "siso_order" must be < "siso_ranking_size".

print("--- FeatBoost Started ---")

# Note: FeatBoost is originally intended for XGBoost Classifier, but now converted to accept XGBoost Regressor.
fs = FeatBoostClassifier(
    estimator=fs_estimator,
    siso_ranking_size=10,
    loss="adaboost",
    metric="acc",
    verbose=2 # 0, 1, or 2
)

# Run feature selection (x_001 means the first column)
fs.fit(X_fs, y_fs)

print("--- FeatBoost Ended ---")
print()
print("FeatBoost - Stopping Condition:")
print(fs.stopping_condition_)
print()
print("FeatBoost - Features Selected:")
print(fs.selected_subset_)
print()
print("FeatBoost - Features Accuracies:")
print(fs.accuracy_)
print()
print("FeatBoost - Feature Importances:")
display(fs.feature_importances_array_)
print()
print("FeatBoost - Feature Rankings:")
print("Note: Each row correpsonds to the number of internal iterations, and each column corresponds to the rank of the feature for that iteration.")
display(fs.siso_ranking_)
print()

# --- New from Edward ---

['lag_1', 'lag_2', 'lag_3', 'lag_24', 'lag_168', 'sin(h)', 'cos(h)', 'is_weekend', 'HourlyDryBulbTemperature', 'HourlyRelativeHumidity', 'HourlyWindSpeed', 'HourlyStationPressure', 'HourlyVisibility']
--- FeatBoost Started ---






Ranking features iteration 01
feature importances of all available feature:
x_001   0.841859
x_007   0.062472
x_004   0.050399
x_009   0.014128
x_006   0.011795
x_003   0.005644
x_005   0.004851
x_002   0.003502
x_010   0.001563
x_011   0.001399
...Evaluating SISO combination 01 which is [np.int64(0)]
Fold 01 accuracy = 0.000000
Fold 02 accuracy = 0.000000
Fold 03 accuracy = 0.000000
Fold 04 accuracy = 0.000000
Fold 05 accuracy = 0.000000
Fold 06 accuracy = 0.001259
Fold 07 accuracy = 0.000000
Fold 08 accuracy = 0.000000
Fold 09 accuracy = 0.000000
Fold 10 accuracy = 0.001259
accuracy for combination 01 is = 0.000252
...Evaluating SISO combination 02 which is [np.int64(6)]
Fold 01 accuracy = 0.000000
Fold 02 accuracy = 0.000000
Fold 03 accuracy = 0.000000
F

array([[8.41859281e-01, 3.50219081e-03, 5.64426370e-03, 5.03989793e-02,
        4.85117314e-03, 1.17948838e-02, 6.24719933e-02, 1.13397243e-03,
        1.41284242e-02, 1.56345696e-03, 1.39899482e-03, 7.60719820e-04,
        4.91740648e-04],
       [8.46731126e-01, 3.12243449e-03, 5.86141180e-03, 4.84337732e-02,
        5.76262223e-03, 1.14236670e-02, 6.08737879e-02, 1.18883094e-03,
        1.30487308e-02, 1.18209596e-03, 1.39420340e-03, 5.66673407e-04,
        4.10658569e-04],
       [8.21874082e-01, 2.07367609e-03, 5.74631896e-03, 6.61979839e-02,
        3.86857498e-03, 1.13900192e-02, 5.91747314e-02, 4.49303258e-03,
        1.43959196e-02, 1.78624957e-03, 3.91670549e-03, 3.01920576e-03,
        2.06338707e-03],
       [8.65837634e-01, 1.89068611e-03, 4.02511610e-03, 3.99638563e-02,
        4.85939672e-03, 1.10001350e-02, 5.34758829e-02, 2.02060235e-03,
        1.01847742e-02, 8.61857436e-04, 1.15756143e-03, 1.84005813e-03,
        2.88249785e-03],
       [8.64581704e-01, 1.90900697e-


FeatBoost - Feature Rankings:
Note: Each row correpsonds to the number of internal iterations, and each column corresponds to the rank of the feature for that iteration.


array([[ 0.,  6.,  3.,  8.,  5.,  2.,  4.,  1.,  9., 10.],
       [ 0.,  6.,  3.,  8.,  5.,  2.,  4.,  1., 10.,  7.],
       [ 0.,  3.,  6.,  8.,  5.,  2.,  7., 10.,  4., 11.],
       [ 0.,  6.,  3.,  5.,  8.,  4.,  2., 12.,  7.,  1.],
       [ 0.,  6.,  3.,  8.,  5.,  2.,  4.,  1.,  9., 10.],
       [ 0.,  6.,  3.,  8.,  5.,  2.,  4., 12.,  7.,  1.],
       [99., 99., 99., 99., 99., 99., 99., 99., 99., 99.],
       [99., 99., 99., 99., 99., 99., 99., 99., 99., 99.],
       [99., 99., 99., 99., 99., 99., 99., 99., 99., 99.],
       [99., 99., 99., 99., 99., 99., 99., 99., 99., 99.]])

# 2. Model Building and Training

### 2.1 Naive & ARX Regression

In [ ]:
# Naive, seasonal naive, and a general ARX-ridge model.
# The naive models choose a previous demand value as the new prediction.
# EVERY other model should do better than them.

naive_rmse = root_mean_squared_error(
    y_test[1:], X_test[1:, allFeats.index("lag_1")] 
)
seasonal_naive_rmse = root_mean_squared_error(
    y_test[24:], X_test[24:, allFeats.index("lag_24")] 
)

print(f"Naive (lag - 1h) RMSE:            {naive_rmse:.3f}")
print(f"Seasonal naive (lag - 24h) RMSE:  {seasonal_naive_rmse:.3f}")

In [ ]:
ARXmodel = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])
ARXmodel.fit(X_train, y_train)
ARXpred = ARXmodel.predict(X_test)
ARXrmse = root_mean_squared_error(y_test, ARXpred)
print(f"ARX-Ridge RMSE: {ARXrmse:.3f}")

In [ ]:
# SARIMAX model
# ! This model is not learning/converging yet

sarimaxIdx = next(i for i, name in enumerate(allFeats) if not name.startswith("lag_"))
sarimax_order = (1, 1, 1)
sarimax_seasonal_order = (1, 1, 1, 24)

SARIMAXmodel = sm.tsa.statespace.SARIMAX(
    endog=y_train,
    exog=X_train[:, sarimaxIdx:],
    order=sarimax_order,
    seasonal_order=sarimax_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    SARIMAXres = SARIMAXmodel.fit(disp=False, maxiter=5, method="lbfgs")

SARIMAXpred = SARIMAXres.get_forecast(
    steps=len(y_test),
    exog=X_test[:, sarimaxIdx:]
).predicted_mean

SARIMAXrmse = root_mean_squared_error(y_test, SARIMAXpred)
print(f"SARIMAX {sarimax_order} x {sarimax_seasonal_order} RMSE: {SARIMAXrmse:.3f}")

### 2.2 Tree-based Models

In [ ]:
# Random Forest

RFmodel = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    random_state=0,
)

RFmodel.fit(X_train, y_train)
RFpred = RFmodel.predict(X_test)
RFrmse = root_mean_squared_error(y_test, RFpred)
print(f"Random Forest RMSE: {RFrmse:.3f}")

In [ ]:
# GBM from sklearn

GBMmodel = GradientBoostingRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    max_features=0.8,
    random_state=0,
)

GBMmodel.fit(X_train, y_train)
GBMpred = GBMmodel.predict(X_test)
GBMrmse = root_mean_squared_error(y_test, GBMpred)
print(f"GBM (sklearn) RMSE: {GBMrmse:.3f}")

In [ ]:
# GBM from xgboost
# !!! For XGBoost and LightGBM, an open source package called "OpenMP" is required.
#     On my Mac, with Homebrew the one-line install command is `brew install libomp`.
#     On PC you'll need to install this through MinGW or Visual Studio.

# XGBmodel = XGBRegressor(
#     n_estimators=400,
#     learning_rate=0.05,
#     max_depth=6,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     objective="reg:squarederror",
#     n_jobs=-1,
#     random_state=0,
# )

# XGBmodel.fit(X_train, y_train)
# XGBpred = XGBmodel.predict(X_test)
# XGBrmse = root_mean_squared_error(y_test, XGBpred)
# print(f"GBM (XGBoost) RMSE: {XGBrmse:.3f}")